# Gradient solvers

> Gradient descent on the actions, with optional constraints (augmented Lagrangian).

In [ ]:
#| default_exp planning.solver.gradient

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import time
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F

from stable_marl.protocols import Constrainable, Costable
from stable_marl.planning.solver.base import SolverBase, gather_agents, prepare_init_action
from stable_marl.planning.solver.categorical import _backward

In [ ]:
#| export
class GradientSolver(SolverBase):
    """
    Gradient descent on continuous actions (the cost must be differentiable in the candidates).
    `num_samples` restarts (the warm start and `var_scale`-perturbed copies) are optimised with
    `optimizer_cls`; the best is kept (per agent in 'per_agent' mode, where each agent's actions only
    receive the gradient of its own cost). Returns ``actions`` ``(n_envs, H, A, action_dim)``.
    """
    discrete = False

    def __init__(self, cost: Costable, n_steps: int, batch_size: int | None = None, var_scale: float = 1.0,
                 num_samples: int = 1, action_noise: float = 0.0,
                 optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD, optimizer_kwargs: dict | None = None,
                 grad_clip: float | None = None, **kwargs):
        super().__init__(cost, batch_size=batch_size, num_samples=num_samples, **kwargs)
        self.n_steps, self.var_scale, self.action_noise = n_steps, var_scale, action_noise
        self.optimizer_cls, self.optimizer_kwargs = optimizer_cls, optimizer_kwargs or {'lr': 1.0}
        self.grad_clip = grad_clip

    def _n_envs_of(self, info_dict: dict) -> int:
        return len(next(v for v in info_dict.values() if isinstance(v, (torch.Tensor, np.ndarray, list))))

    def init_action(self, n_envs: int, info_dict: dict, init_action: torch.Tensor | None) -> torch.Tensor:
        "The restarts (n_envs, num_samples, H, A, action_dim): the warm start, then perturbed copies."
        with torch.no_grad():
            actions = prepare_init_action(self.cost, info_dict, init_action, self.horizon, n_envs,
                                          self._num_agents, self.action_dim, device=self.device).to(self.dtype)
            actions = actions.unsqueeze(1).repeat_interleave(self.num_samples, dim=1)
            actions[:, 1:] += torch.randn(actions[:, 1:].shape, generator=self.torch_gen, device=self.device,
                                          dtype=self.dtype) * self.var_scale
        return actions

    def _step(self, x: torch.Tensor, optim: torch.optim.Optimizer):
        if self.grad_clip is not None:
            torch.nn.utils.clip_grad_norm_([x], self.grad_clip)
        optim.step()
        optim.zero_grad(set_to_none=True)
        if self.action_noise > 0:
            with torch.no_grad():
                x += torch.randn(x.shape, generator=self.torch_gen, device=self.device, dtype=self.dtype) * self.action_noise

    def _best(self, infos: dict, x: torch.Tensor, B: int) -> tuple[torch.Tensor, torch.Tensor]:
        "The best restart (per agent): actions (B, H, A, action_dim), costs (B, A)."
        with torch.no_grad():
            costs = self._costs(self.cost.get_cost(dict(infos), x), B, self.num_samples)
            best = costs.argmin(dim=1, keepdim=True)
            return gather_agents(x.detach(), best)[:, 0], costs.gather(1, best)[:, 0]

    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs = self._n_envs_of(info_dict)
        params, actions, final_costs = self.init_action(n_envs, info_dict, init_action), [], []
        for start, end in self._batches(n_envs):
            B, infos = end - start, self._expand(info_dict, start, end)
            x = params[start:end].clone().requires_grad_(True)
            optim = self.optimizer_cls([x], **self.optimizer_kwargs)
            for _ in range(self.n_steps):
                costs = self._costs(self.cost.get_cost(dict(infos), x), B, self.num_samples)
                if not costs.requires_grad:
                    raise RuntimeError(f"{type(self).__name__} needs a cost differentiable in the candidates")
                _backward(costs, x, self.mode)
                self._step(x, optim)
            plan, cost = self._best(infos, x, B)
            actions.append(plan.cpu())
            final_costs += self._cost_list(cost)
        self._log(start_time)
        return {'actions': torch.cat(actions), 'costs': final_costs}

In [ ]:
#| export
class LagrangianSolver(GradientSolver):
    """
    Gradient descent on the augmented Lagrangian of a cost with inequality constraints
    (:class:`Constrainable`, ``g <= 0`` when satisfied): `n_outer_steps` rounds of `n_steps`
    gradient steps, after each of which the multipliers grow with the violations and the penalty
    `rho` is multiplied by `rho_scale` (up to `rho_max`). Without constraints, plain gradient descent.

    Constraints are ``(B, S, C)``; in 'per_agent' mode they may also be per agent,
    ``(B, S, A, C)``, and each agent's actions follow its own cost and constraint terms (joint
    constraints apply to every agent). Multipliers persist across solves (`persist_multipliers`).
    """
    def __init__(self, cost: Costable, n_steps: int, n_outer_steps: int = 5, rho_init: float = 1.0,
                 rho_max: float = 1e4, rho_scale: float = 2.0, persist_multipliers: bool = True,
                 optimizer_cls: type[torch.optim.Optimizer] = torch.optim.Adam, **kwargs):
        super().__init__(cost, n_steps=n_steps, optimizer_cls=optimizer_cls, **kwargs)
        self.n_outer_steps, self.rho_init, self.rho_max, self.rho_scale = n_outer_steps, rho_init, rho_max, rho_scale
        self.persist_multipliers = persist_multipliers
        self._lambdas: torch.Tensor | None = None

    def _constraints(self, infos: dict, x: torch.Tensor, B: int) -> torch.Tensor | None:
        """
        Constraint values, or None if the cost has none: joint mode ``(B, S, C)`` (per-agent
        constraints flattened into joint ones), 'per_agent' mode ``(B, S, A, C)`` (joint ones shared).
        """
        if not isinstance(self.cost, Constrainable):
            return None
        g = self.cost.get_constraints(dict(infos), x)
        S, A = self.num_samples, self._num_agents
        if g.ndim == 4 and g.shape[:3] == (B, S, A):
            return g.flatten(2) if self.mode == 'joint' else g
        if g.ndim == 3 and g.shape[:2] == (B, S):
            return g if self.mode == 'joint' else g.unsqueeze(2).expand(B, S, A, g.shape[-1])
        raise ValueError(f"Constraints must be (B, S, C) or (B, S, A, C), got {tuple(g.shape)}")

    def _loss(self, costs: torch.Tensor, g: torch.Tensor | None, lambdas: torch.Tensor | None, rho: float) -> torch.Tensor:
        "Augmented Lagrangian (B, S, A): per agent, or the joint one shared by the agents."
        if g is None:
            return costs
        penalty = (lambdas[:, None] * g).sum(-1) + rho * F.relu(g).pow(2).sum(-1)    # (B, S) or (B, S, A)
        return costs + (penalty.unsqueeze(-1) if self.mode == 'joint' else penalty)

    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs = self._n_envs_of(info_dict)
        params = self.init_action(n_envs, info_dict, init_action)
        if not self.persist_multipliers or (self._lambdas is not None and self._lambdas.shape[0] != n_envs):
            self._lambdas = None
        outputs = {'costs': [], 'constraint_violation': []}
        actions = []
        for start, end in self._batches(n_envs):
            B, infos, rho = end - start, self._expand(info_dict, start, end), self.rho_init
            x = params[start:end].clone().requires_grad_(True)
            g = None
            for _ in range(self.n_outer_steps):
                optim = self.optimizer_cls([x], **self.optimizer_kwargs)
                for _ in range(self.n_steps):
                    costs = self._costs(self.cost.get_cost(dict(infos), x), B, self.num_samples)
                    g = self._constraints(infos, x, B)
                    if g is not None and self._lambdas is None:
                        shape = (n_envs, g.shape[-1]) if self.mode == 'joint' else (n_envs, self._num_agents, g.shape[-1])
                        self._lambdas = torch.zeros(shape, dtype=self.dtype, device=self.device)
                    lambdas = None if g is None else self._lambdas[start:end]
                    _backward(self._loss(costs, g, lambdas, rho), x, self.mode)
                    self._step(x, optim)
                if g is not None:
                    with torch.no_grad():
                        g = self._constraints(infos, x, B)
                        self._lambdas[start:end] = (self._lambdas[start:end] + rho * g.mean(dim=1)).clamp(min=0.0)
                        rho = min(self.rho_max, rho * self.rho_scale)
            if g is not None:
                outputs['constraint_violation'].append(F.relu(g).mean().item())
            plan, cost = self._best(infos, x, B)
            actions.append(plan.cpu())
            outputs['costs'] += self._cost_list(cost)
        outputs['actions'] = torch.cat(actions)
        outputs['lambdas'] = None if self._lambdas is None else self._lambdas.cpu()
        self._log(start_time)
        return outputs

### Tests

In [ ]:
from types import SimpleNamespace
import gymnasium as gym
from stable_marl.planning.solver.base import Solver, PLANNING_MODES

class QuadraticCost:
    "Squared distance to a target plan (H, A, X); optional constraint: every action <= `cap`."
    def __init__(self, target, per_agent=False, cap=None):
        self.target, self.per_agent, self.cap = target, per_agent, cap
        if cap is not None:
            self.get_constraints = lambda info_dict, candidates: (candidates - cap).amax(dim=(2, 4)).unsqueeze(-1)  # (B, S, A, 1)
    def get_cost(self, info_dict, candidates):
        cost = (candidates - self.target).pow(2).sum(dim=(2, 4))
        return cost if self.per_agent else cost.sum(-1)
    def criterion(self, info_dict, candidates):
        return self.get_cost(info_dict, candidates)

cfg = SimpleNamespace(horizon=3, action_block=1)
space = gym.spaces.Box(-1, 1, (2, 2))
target = torch.tensor([[0.5, -0.3], [0.1, 0.8]]).repeat(3, 1, 1)       # (H, A=2, 2)
for mode in PLANNING_MODES:
    cost = QuadraticCost(target, per_agent=mode == 'per_agent')
    for solver in (GradientSolver(cost, n_steps=60, num_samples=3, var_scale=0.3, optimizer_kwargs={'lr': 0.1}, mode=mode),
                   LagrangianSolver(cost, n_steps=50, n_outer_steps=2, optimizer_kwargs={'lr': 0.05}, mode=mode)):
        solver.configure(action_space=space, n_envs=2, config=cfg)
        assert isinstance(solver, Solver)
        out = solver.solve({'x': torch.zeros(2, 1)})
        err = (out['actions'] - target).abs().max().item()
        assert out['actions'].shape == (2, 3, 2, 2) and err < 0.1, (type(solver).__name__, mode, err)

    # with the constraint `action <= 0.2`, the solution moves to the cap where the target is above it
    capped = QuadraticCost(target, per_agent=mode == 'per_agent', cap=0.2)
    solver = LagrangianSolver(capped, n_steps=40, n_outer_steps=6, optimizer_kwargs={'lr': 0.05}, mode=mode)
    solver.configure(action_space=space, n_envs=2, config=cfg)
    out = solver.solve({'x': torch.zeros(2, 1)})
    assert out['actions'].max() < 0.25 and out['lambdas'] is not None, (mode, out['actions'].max())
    assert (out['actions'][..., 0, 1] - (-0.3)).abs().max() < 0.1           # a coordinate below the cap is unaffected

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()